In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.1108434
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5127
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
g2/2/pi*1e3

In [ ]:
t_max = 60/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50* ω_a) # Time step for the simulation

dt_save=1/(sqrt(2)*ω_p)
t_save = max(t_min_save,0):dt_save:t_max


ϕ_b_0, q_b_0=0.0, 0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, 0.0, ϕ_b_0, q_b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,11)
Re_a_list=range(-10,10,25)
Final_all = Dict{Float64,Tuple{Dict{Tuple{Float64,Float64}, Vector{Float64}},Dict{Tuple{Float64,Float64}, Vector{Float64}}}}()
ϵ_d_g2_ratio_list = 1:1:15

Threads.@threads for ϵ_d_g2_ratio in ϵ_d_g2_ratio_list
    ϵ_d=ϵ_d_g2_ratio*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a_re = Dict{Tuple{Float64,Float64}, Vector{Float64}}()
    Final_mode_a_im = Dict{Tuple{Float64,Float64}, Vector{Float64}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        ϕ_a_0_i=sqrt(2*Z_a)*re_i
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            q_a_0_j = sqrt(2/Z_a)*im_j

            system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0_i, q_a_0_j, ϕ_b_0, q_b_0)
            system_prepared = prepare(system)

            u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            ϕ_a, _, q_a, _, t_list = default_solver((u_dot_0, u_0), (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Re_rotating = @.(ϕ_a * cos(ω_a * t_list) + q_a * Z_a * sin(-ω_a * t_list)) / sqrt(2*Z_a)
            Im_rotating = @.(-ϕ_a/Z_a * sin(-ω_a * t_list) + q_a * cos(ω_a * t_list)) * sqrt(Z_a/2)

            Final_mode_a_re[(re_i,im_j)] = Re_rotating
            Final_mode_a_im[(re_i,im_j)] = Im_rotating
        end
    end
    Final_all[ϵ_d_g2_ratio] = (Final_mode_a_re, Final_mode_a_im)
end

In [ ]:
for ϵ_d_g2_ratio in ϵ_d_g2_ratio_list
    Final_mode_a_re, Final_mode_a_im = Final_all[ϵ_d_g2_ratio]
    p0 =plot()
    for (re_i, im_j) in keys(Final_mode_a_re)
        scatter!(p0, Final_mode_a_re[(re_i, im_j)], Final_mode_a_im[(re_i, im_j)], zcolor=t_list, label="", color=:black,
        ms=1)
    end
    plot!(p0, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"|\alpha|^2="*string(round(ϵ_d_g2_ratio/2,digits=3)))
    plot!(colorbar_title="\n"*L"κ_bt")
    display(plot(p0,
    xlims=(-15,15),
    ylims=(-6,6),top_margin=5mm))
end

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)
Final_all = Dict{Float64,Tuple{Dict{Tuple{Float64,Float64}, Vector{Float64}},Dict{Tuple{Float64,Float64}, Vector{Float64}}}}()
ϵ_d_g2_ratio_list = 7:0.2:9

Threads.@threads for ϵ_d_g2_ratio in ϵ_d_g2_ratio_list
    ϵ_d=ϵ_d_g2_ratio*g2
    drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a_re = Dict{Tuple{Float64,Float64}, Vector{Float64}}()
    Final_mode_a_im = Dict{Tuple{Float64,Float64}, Vector{Float64}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        ϕ_a_0_i=sqrt(2*Z_a)*re_i
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            q_a_0_j = sqrt(2/Z_a)*im_j

            system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0_i, q_a_0_j, ϕ_b_0, q_b_0)
            system_prepared = prepare(system)

            u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            ϕ_a, _, q_a, _, t_list = default_solver(u_dot_0, u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Re_rotating = @.(ϕ_a * cos(ω_a * t_list) + q_a * Z_a * sin(-ω_a * t_list)) / sqrt(2*Z_a)
            Im_rotating = @.(-ϕ_a/Z_a * sin(-ω_a * t_list) + q_a * cos(ω_a * t_list)) * sqrt(Z_a/2)

            Final_mode_a_re[(re_i,im_j)] = Re_rotating
            Final_mode_a_im[(re_i,im_j)] = Im_rotating
        end
    end
    Final_all[ϵ_d_g2_ratio] = (Final_mode_a_re, Final_mode_a_im)
end

In [ ]:
for ϵ_d_g2_ratio in ϵ_d_g2_ratio_list
    Final_mode_a_re, Final_mode_a_im = Final_all[ϵ_d_g2_ratio]
    p0 = plot()
    for (re_i, im_j) in keys(Final_mode_a_re)
        scatter!(p0, Final_mode_a_re[(re_i, im_j)], Final_mode_a_im[(re_i, im_j)], label="", color=:black,
        ms=1)
    end
    plot!(p0, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"|\alpha|^2="*string(round(ϵ_d_g2_ratio/2,digits=3)))
    plot!(colorbar_title="\n"*L"κ_bt")
    display(plot(p0,
    xlims=(-15,15),
    ylims=(-6,6),top_margin=5mm))
end